In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nouurmohaamed/male-voice-1/a1198aa0-4029-435d-ab16-00261d4a924e.mp3


# Models used :
**1. "facebook/dinov2-base" ----------> create image embeddings**
2. 

In [2]:
!pip install fastapi uvicorn pyngrok 

In [3]:
!pip uninstall -y transformers -q
!pip install -U git+https://github.com/huggingface/transformers accelerate -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 6.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 12.9 MB/s eta 0:00:0000:01


In [4]:
!pip install kokoro soundfile
!apt-get -qq -y install espeak-ng

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 6.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 

In [5]:
!pip install sentencepiece  # for "facebook/nllb-200-distilled-600M" since models from Meta were trained with a SentencePiece tokenizer, so you need the sentencepiece package installed.

In [6]:
import torch

# generate text

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [27]:
def generate_text(prompt, max_length=300):
    messages=[
        {
            "role": "user", 
            "content":prompt
        }
    ]
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)
    print("Input tokens:", inputs.input_ids.shape[1])
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.5,
            pad_token_id=tokenizer.eos_token_id
        )

    outputs = outputs[0][inputs.input_ids.shape[1]:]
    generated_response=tokenizer.batch_decode(
        [outputs],
        skip_special_tokens=True
    )[0]
    return generated_response

INFO:     156.205.104.206:0 - "POST /embedd HTTP/1.1" 200 OK
Input tokens: 2089
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 2618
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/spacy/util.py:1800: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


INFO:     156.205.104.206:0 - "POST /speech HTTP/1.1" 200 OK
Input tokens: 1921
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 1975
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 2180
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 2009
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 2061
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 2061
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 1921
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 1925
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
Input tokens: 1922
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     152.55.177.144:0 - "POST /embedd HTTP/1.1" 401 Unauthorized
INFO:     152.55.177.144:0 - "POST /embedd HTTP/1.1" 401 Unauthorized
INFO:     152.55.177.152:0 - "POST /embedd HTTP/1.1" 200 OK
I

# text to speech

In [9]:
from kokoro import KPipeline
import soundfile as sf #convert the generated array of audio samples to sound file


In [10]:
from fastapi.responses import FileResponse
def generate_speech(text,language):
    pipeline = KPipeline(lang_code=language) #create the model
    generator = pipeline( 
    text,
    voice="af_heart"
    )
    chunks = []

    for _, _, audio in generator:
        chunks.append(audio)

    full_audio = np.concatenate(chunks)

    sf.write("answer.wav", full_audio, 24000)
    return FileResponse(
            path="answer.wav",
            media_type="audio/wav",
            filename="answer.wav"
        )


# translation 

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translation_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
translation_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", device_map="auto")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [12]:
def generate_translation(text,target_lang,src_lang):
    translation_tokenizer.src_lang=src_lang
    input_tokens=translation_tokenizer(text,return_tensors="pt").to(translation_model.device)
    output_tokens=translation_model.generate(**input_tokens,max_new_tokens=900,forced_bos_token_id=translation_tokenizer.convert_tokens_to_ids(target_lang))
    translation = translation_tokenizer.batch_decode(
    output_tokens,
    skip_special_tokens=True
)[0]
    print(translation)
    return translation
    

# create image embedding

In [13]:
from PIL import Image
from transformers import AutoImageProcessor
from transformers import AutoModel
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(
    "facebook/dinov2-base"
)

embedding_model = AutoModel.from_pretrained(
    "facebook/dinov2-base"
).to(device)

embedding_model.eval()

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

In [14]:
import base64
import io
from PIL import Image

def base64_to_image(image_base64: str): #return the image from string form to image object

    image_bytes = base64.b64decode(image_base64)

    buffer = io.BytesIO(image_bytes)

    image = Image.open(buffer).convert("RGB") #return to image object

    return image

In [15]:
def embed_image(image_base64):
    image=base64_to_image(image_base64)
    inputs = processor(  #return a pytorch representing the pixel values of image "process the image"
        images=image,
        return_tensors="pt"
    ).to(device)
# inputs: image object ----> matrix of pixels each pixel represented by 3 channels "RGB"
    with torch.no_grad():

        outputs = embedding_model(**inputs) #create the embedding vector that represent the image

    embedding = outputs.last_hidden_state[:,0]

    embedding = embedding / embedding.norm(dim=-1, keepdim=True) #normalize the embedding vector
    embedding=embedding.cpu().numpy()[0]

    return embedding.tolist() #return a python list to transfere acrous the network without errors

# server backbone

In [16]:
from fastapi import FastAPI,Request,HTTPException
import uvicorn,socket
import threading,time
from pyngrok import ngrok,conf

In [17]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("NGROK_API")
secret_value_1 = user_secrets.get_secret("server_authentication_password")

In [18]:
app=FastAPI()

In [19]:

@app.post("/generate")
async def handle_req(req:Request):
    if req.headers['authorization']!= secret_value_1:
        raise HTTPException(status_code=401,detail="unauthorized")
    data= await req.json()
    return{
        'response':generate_text(
            data.get('prompt',''),
            data.get('max_length',300)
        )
    }

In [20]:

@app.post("/embedd")
async def handle_req(req:Request):
    if req.headers['authorization']!= secret_value_1:
        raise HTTPException(status_code=401,detail="unauthorized")
    data= await req.json()
    return{
        'response':embed_image(
            data.get('image','')
        )
    }

In [21]:
@app.post("/speech")
async def handle_req(req:Request):
    if req.headers['authorization']!= secret_value_1:
        raise HTTPException(status_code=401,detail="unauthorized")
    data= await req.json()
    return generate_speech(
            data.get('text','no text provided to turn into speech'),
            data.get('language','a')
                                  )

In [22]:
@app.post("/translate")
async def handle_req(req:Request):
    if req.headers['authorization']!= secret_value_1:
        raise HTTPException(status_code=401,detail="unauthorized")
    data= await req.json()
    return{
        'response':generate_translation(
            data.get('text',''),
            data.get('target_language','eng_Latn'),
            data.get('src_language','eng_Latn')
        )
    }

In [23]:
# create a new port number for kaggle notebook
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port
    
port = free_port()

In [24]:
#create the endpoint "URL"
conf.get_default().auth_token = secret_value_0 #use my account
public_url = ngrok.connect(port).public_url #connect this kaggle notebook with ngrok and return url: The public URL is the address of your server on the internet.
print("Your public URL:", public_url)

Your public URL: https://copied-hardcore-ensure.ngrok-free.dev                                      


In [25]:
#run the server "waiting in a loop for requests"
def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:48715 (Press CTRL+C to quit)


INFO:     156.205.104.206:0 - "POST /embedd HTTP/1.1" 200 OK
Input tokens: 2089
INFO:     156.205.104.206:0 - "POST /generate HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error